In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import KFold, train_test_split
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout, BatchNormalization
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras import initializers
SEED=42; IMG_SIZE=224; NUM_CLASSES=37; BATCH_SIZE=32; EPOCHS_HEAD=10; EPOCHS_FINE=5
BREEDS=["Abyssinian","Bengal","Birman","Bombay","British Shorthair","Egyptian Mau","Maine Coon","Persian","Ragdoll","Russian Blue","Siamese","Sphynx","american_bulldog","american_pit_bull_terrier","basset_hound","beagle","boxer","chihuahua","english_cocker_spaniel","english_setter","german_shorthaired","great_pyrenees","havanese","japanese_chin","keeshond","leonberger","miniature_pinscher","newfoundland","pomeranian","pug","saint_bernard","samoyed","scottish_terrier","shiba_inu","staffordshire_bull_terrier","wheaten_terrier","yorkshire_terrier"]
np.random.seed(SEED); tf.random.set_seed(SEED)


## Dataset Loading (Oxford-IIIT Pet, 224x224x3)

In [ ]:
def preprocess(image, label):
    image=tf.image.resize(image,(IMG_SIZE,IMG_SIZE))
    image=tf.cast(image,tf.float32)/127.5-1.0
    return image,label
def load_pet():
    ds_train=tfds.load('oxford_iiit_pet',split='train',as_supervised=True,data_dir='/Users/vijay/tensorflow_datasets')
    ds_test=tfds.load('oxford_iiit_pet',split='test',as_supervised=True,data_dir='/Users/vijay/tensorflow_datasets')
    def to_arrays(ds):
        imgs,labs=[],[]
        for im,lb in tfds.as_numpy(ds):
            imgs.append(im); labs.append(int(lb))
        return imgs, np.array(labs)
    return to_arrays(ds_train), to_arrays(ds_test)
(train_images,train_labels),(test_images,test_labels)=load_pet()
print("train",len(train_images),"test",len(test_images))


In [ ]:
from sklearn.model_selection import train_test_split
idx_train,idx_val,_,_=train_test_split(np.arange(len(train_labels)),train_labels,test_size=0.15,random_state=SEED,stratify=train_labels)
train_images_split=[train_images[i] for i in idx_train]; train_labels_split=train_labels[idx_train]
val_images=[train_images[i] for i in idx_val]; val_labels=train_labels[idx_val]
print(f"split train {len(train_images_split)} val {len(val_images)} test {len(test_images)}")


## Sample Images

In [ ]:
fig,axes=plt.subplots(2,5,figsize=(12,5))
for ax,img,lab in zip(axes.flat,train_images_split[:10],train_labels_split[:10]):
    ax.imshow(img); ax.set_title(BREEDS[int(lab)],fontsize=7); ax.axis('off')
fig.suptitle("Sample Oxford-IIIT Pet images (224x224)")
plt.tight_layout(); plt.savefig("exp5_plot_01_sample_images.png",dpi=130); plt.show()


## Class Distribution

In [ ]:
counts=np.bincount(train_labels,minlength=NUM_CLASSES)
fig,ax=plt.subplots(figsize=(12,4))
ax.bar(range(NUM_CLASSES),counts,color='steelblue',edgecolor='black')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels([b[:8] for b in BREEDS],rotation=45,ha='right',fontsize=6)
ax.set_title("Class distribution of Oxford-IIIT Pet training set (3680 images, 37 breeds)")
ax.set_ylabel("count"); plt.tight_layout(); plt.savefig("exp5_plot_02_class_distribution.png",dpi=130); plt.show()


## Feature Extraction with MobileNetV2

In [ ]:
def make_tfds(images,labels,shuffle=False,batch=BATCH_SIZE):
    def gen():
        for im,lb in zip(images,labels):
            yield im,lb
    ds=tf.data.Dataset.from_generator(lambda: gen(),output_signature=(tf.TensorSpec(shape=(None,None,3),dtype=tf.uint8),tf.TensorSpec(shape=(),dtype=tf.int64)))
    ds=ds.map(preprocess,num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds=ds.shuffle(1024,seed=SEED)
    ds=ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds
def build_base():
    return MobileNetV2(include_top=False,weights='imagenet',input_shape=(IMG_SIZE,IMG_SIZE,3))
base=build_base()
extractor=Model(base.input,GlobalAveragePooling2D()(base.output))
train_ds=make_tfds(train_images_split,train_labels_split)
val_ds=make_tfds(val_images,val_labels)
test_ds=make_tfds(test_images,test_labels)
train_features=extractor.predict(train_ds,verbose=1)
val_features=extractor.predict(val_ds,verbose=1)
test_features=extractor.predict(test_ds,verbose=1)
print(train_features.shape,val_features.shape,test_features.shape)
y_train_oh=tf.keras.utils.to_categorical(train_labels_split,NUM_CLASSES)
y_val_oh=tf.keras.utils.to_categorical(val_labels,NUM_CLASSES)
y_test_oh=tf.keras.utils.to_categorical(test_labels,NUM_CLASSES)


## Helper: Head Model

In [ ]:
def build_head(input_dim,num_classes,initializer='he',dropout=0.0,l2_reg=0.0,use_bn=False):
    m={"zeros":initializers.Zeros(),"random":initializers.RandomNormal(0,0.05),"glorot":initializers.GlorotUniform(),"he":initializers.HeNormal()}[initializer]
    reg=l2(l2_reg) if l2_reg>0 else None
    model=Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(128,activation='relu',kernel_initializer=m,kernel_regularizer=reg))
    if use_bn: model.add(BatchNormalization())
    if dropout>0: model.add(Dropout(dropout))
    model.add(Dense(num_classes,activation='softmax',kernel_initializer=m))
    return model
def train_head(tr_f,tr_oh,va_f,va_oh,initializer='he',dropout=0.0,l2_reg=0.0,use_bn=False,optimizer_name='adam',lr=0.001,epochs=EPOCHS_HEAD,batch_size=BATCH_SIZE):
    model=build_head(tr_f.shape[1],NUM_CLASSES,initializer,dropout,l2_reg,use_bn)
    opt={"sgd":tf.keras.optimizers.SGD(lr),"momentum":tf.keras.optimizers.SGD(lr,momentum=0.9),"rmsprop":tf.keras.optimizers.RMSprop(lr),"adam":tf.keras.optimizers.Adam(lr)}[optimizer_name]
    model.compile(optimizer=opt,loss='categorical_crossentropy',metrics=['accuracy'])
    import time
    s=time.perf_counter(); hist=model.fit(tr_f,tr_oh,validation_data=(va_f,va_oh),epochs=epochs,batch_size=batch_size,verbose=0); t=time.perf_counter()-s
    return model,hist,t
def save_fig(fig,name):
    fig.tight_layout(); fig.savefig(name,dpi=130,bbox_inches='tight'); plt.close(fig)


## 5. Weight Initialization (Zero, Random, Xavier, He)

In [ ]:
inits=['zeros','random','glorot','he']; labels=['Zero','Random','Xavier/Glorot','He']
hists=[]; times=[]
for init in inits:
    _,h,t=train_head(train_features,y_train_oh,val_features,y_val_oh,initializer=init,epochs=10,verbose=0)
    hists.append(h); times.append(t); print(init, f"{max(h.history['val_accuracy']):.4f}")
fig,ax=plt.subplots(figsize=(9,5))
for h,lab,c in zip(hists,labels,['steelblue','darkorange','seagreen','crimson']):
    ax.plot(h.history['loss'],label=lab,color=c)
ax.set_title("Plot 1: Training Loss vs Epoch for Different Initializations"); ax.set_xlabel("epoch"); ax.set_ylabel("training loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_01_init_loss.png",dpi=130); plt.show()
fig,ax=plt.subplots(figsize=(9,5))
for h,lab,c in zip(hists,labels,['steelblue','darkorange','seagreen','crimson']):
    ax.plot(np.array(h.history['val_accuracy'])*100,label=lab,color=c)
ax.set_title("Plot 2: Validation Accuracy vs Epoch for Different Initializations"); ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_02_init_valacc.png",dpi=130); plt.show()


## 6. Regularization and Overfitting

In [ ]:
regs=[("No regularization",0.0,0.0,False),("L2 (0.01)",0.0,0.01,False),("Dropout 0.5",0.5,0.0,False),("BatchNorm",0.0,0.0,True)]
rh=[]; rlabels=[]
for name,drop,l2v,bn in regs:
    _,h,_=train_head(train_features,y_train_oh,val_features,y_val_oh,dropout=drop,l2_reg=l2v,use_bn=bn,epochs=10,verbose=0)
    rh.append(h); rlabels.append(name); print(name, f"{max(h.history['val_accuracy']):.4f}")
h0=rh[0]
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(np.array(h0.history['accuracy'])*100,label='training accuracy',color='steelblue')
ax.plot(np.array(h0.history['val_accuracy'])*100,label='validation accuracy',color='darkorange')
ax.set_title("Plot 3: Training and Validation Accuracy (No Regularization)"); ax.set_xlabel("epoch"); ax.set_ylabel("accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_03_reg_accuracy.png",dpi=130); plt.show()
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(h0.history['loss'],label='training loss',color='steelblue')
ax.plot(h0.history['val_loss'],label='validation loss',color='darkorange')
ax.set_title("Plot 4: Training and Validation Loss (No Regularization)"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_04_reg_loss.png",dpi=130); plt.show()


## 7. Batch Normalization (Numerical Example and Plot 5)

In [ ]:
# numerical example x=[2,4,6,8] mean 5 var 5 normalized [-1.342,-0.447,0.447,1.342]
x=np.array([2,4,6,8],dtype=float); mu=x.mean(); var=((x-mu)**2).mean(); std=np.sqrt(var)
x_hat=(x-mu)/std; print("mu",mu,"var",var,"x_hat",x_hat)
# plot With vs Without BN
h_bn=rh[3]; h_nobn=rh[0]
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(np.array(h_nobn.history['val_accuracy'])*100,label='Without BN',color='steelblue')
ax.plot(np.array(h_bn.history['val_accuracy'])*100,label='With BN',color='seagreen')
ax.set_title("Plot 5: With vs Without Batch Normalization"); ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_05_bn_comparison.png",dpi=130); plt.show()


## 8. Optimization Algorithms (SGD, Momentum, RMSProp, Adam)

In [ ]:
opts=['sgd','momentum','rmsprop','adam']; olabels=['SGD','SGD+Momentum','RMSProp','Adam']
oh=[]; ot=[]
for o in opts:
    _,h,t=train_head(train_features,y_train_oh,val_features,y_val_oh,optimizer_name=o,dropout=0.25,epochs=10,verbose=0)
    oh.append(h); ot.append(t); print(o, f"{max(h.history['val_accuracy']):.4f}")
fig,ax=plt.subplots(figsize=(9,5))
for h,lab,c in zip(oh,olabels,['steelblue','darkorange','seagreen','crimson']):
    ax.plot(h.history['loss'],label=lab,color=c)
ax.set_title("Plot 6: Training Loss vs Epoch for Different Optimizers"); ax.set_xlabel("epoch"); ax.set_ylabel("training loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_06_optimizer_loss.png",dpi=130); plt.show()
fig,ax=plt.subplots(figsize=(9,5))
for h,lab,c in zip(oh,olabels,['steelblue','darkorange','seagreen','crimson']):
    ax.plot(np.array(h.history['val_accuracy'])*100,label=lab,color=c)
ax.set_title("Plot 7: Validation Accuracy vs Epoch for Different Optimizers"); ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_07_optimizer_valacc.png",dpi=130); plt.show()
for lab,h,t in zip(olabels,oh,ot):
    print(f"{lab}: loss {h.history['loss'][-1]:.4f} best_val {max(h.history['val_accuracy']):.4f} time {t:.1f}s")


## 9. Hyperparameter Tuning (LR, Batch Size, Dropout)

In [ ]:
lrs=[0.001,0.0001]; lr_h=[]
for lr in lrs:
    _,h,_=train_head(train_features,y_train_oh,val_features,y_val_oh,dropout=0.25,optimizer_name='adam',lr=lr,epochs=10,verbose=0)
    lr_h.append(h)
fig,ax=plt.subplots(figsize=(8,5))
vals=[max(h.history['val_accuracy'])*100 for h in lr_h]
ax.bar([str(l) for l in lrs],vals,color=['steelblue','seagreen'],edgecolor='black')
ax.set_title("Plot 8: Learning Rate vs Validation Accuracy"); ax.set_xlabel("learning rate"); ax.set_ylabel("best validation accuracy (%)")
for i,v in enumerate(vals): ax.text(i,v+0.3,f"{v:.1f}%",ha='center')
plt.tight_layout(); plt.savefig("exp5_plot_08_lr_vs_acc.png",dpi=130); plt.show()
bss=[16,32,64]; bs_h=[]
for bs in bss:
    _,h,_=train_head(train_features,y_train_oh,val_features,y_val_oh,dropout=0.25,batch_size=bs,epochs=10,verbose=0)
    bs_h.append(h)
fig,ax=plt.subplots(figsize=(8,5))
vals=[max(h.history['val_accuracy'])*100 for h in bs_h]
ax.bar([str(b) for b in bss],vals,color=['steelblue','seagreen','darkorange'],edgecolor='black')
ax.set_title("Plot 9: Batch Size vs Validation Accuracy"); ax.set_xlabel("batch size"); ax.set_ylabel("best validation accuracy (%)")
for i,v in enumerate(vals): ax.text(i,v+0.3,f"{v:.1f}%",ha='center')
plt.tight_layout(); plt.savefig("exp5_plot_09_bs_vs_acc.png",dpi=130); plt.show()
drops=[0.0,0.25,0.5]; d_h=[]
for d in drops:
    _,h,_=train_head(train_features,y_train_oh,val_features,y_val_oh,dropout=d,epochs=10,verbose=0)
    d_h.append(h)
fig,ax=plt.subplots(figsize=(8,5))
vals=[max(h.history['val_accuracy'])*100 for h in d_h]
ax.bar([str(d) for d in drops],vals,color=['steelblue','seagreen','darkorange'],edgecolor='black')
ax.set_title("Plot 10: Dropout Rate vs Validation Accuracy"); ax.set_xlabel("dropout rate"); ax.set_ylabel("best validation accuracy (%)")
for i,v in enumerate(vals): ax.text(i,v+0.3,f"{v:.1f}%",ha='center')
plt.tight_layout(); plt.savefig("exp5_plot_10_dropout_vs_acc.png",dpi=130); plt.show()
# convolution output size example O = floor((N+2P-K)/S)+1
def conv_out(N,K,P,S): return (N+2*P-K)//S+1
print("224 with 3x3 same s1:", conv_out(224,3,1,1), "s2:", conv_out(224,3,1,2))


## 10. Transfer Learning and Fine-Tuning (MobileNetV2)

In [ ]:
base_ft=build_base(); base_ft.trainable=False
inputs=Input(shape=(IMG_SIZE,IMG_SIZE,3))
x=base_ft(inputs,training=False); x=GlobalAveragePooling2D()(x); x=Dense(128,activation='relu')(x); x=Dropout(0.25)(x)
outputs=Dense(NUM_CLASSES,activation='softmax')(x)
frozen_model=Model(inputs,outputs)
frozen_model.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
frozen_hist=frozen_model.fit(train_ds,validation_data=val_ds,epochs=10,verbose=2)
# fine-tune last 20 layers excluding BN
for l in base_ft.layers: l.trainable=False
for l in base_ft.layers[-20:]:
    if not isinstance(l,BatchNormalization): l.trainable=True
frozen_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
fine_hist=frozen_model.fit(train_ds,validation_data=val_ds,epochs=5,verbose=2)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(np.array(frozen_hist.history['val_accuracy'])*100,label='Feature Extraction (frozen)',color='steelblue')
ax.plot(np.array(fine_hist.history['val_accuracy'])*100,label='Fine-Tuning (unfrozen last 20)',color='seagreen')
ax.set_title("Plot 11: Feature Extraction vs Fine-Tuning"); ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_11_fe_vs_fine.png",dpi=130); plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(frozen_hist.history['loss'],label='train loss',color='steelblue'); axes[0].plot(frozen_hist.history['val_loss'],label='val loss',color='darkorange')
axes[0].set_title("Loss - Feature Extraction"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(fine_hist.history['loss'],label='train loss',color='seagreen'); axes[1].plot(fine_hist.history['val_loss'],label='val loss',color='crimson')
axes[1].set_title("Loss - Fine Tuning"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("exp5_plot_12_loss_finetune.png",dpi=130); plt.show()
print("Frozen best val",max(frozen_hist.history['val_accuracy']), "Fine best val",max(fine_hist.history['val_accuracy']))


## 11. 5-Fold Cross-Validation

In [ ]:
all_feat=np.concatenate([train_features,val_features],axis=0)
all_lab=np.concatenate([train_labels_split,val_labels])
all_oh=tf.keras.utils.to_categorical(all_lab,NUM_CLASSES)
configs=[("C1: Adam lr0.001 dropout0.25 bs32",{'dropout':0.25,'optimizer_name':'adam','lr':0.001}),("C2: Adam lr0.0001 dropout0.25",{'dropout':0.25,'optimizer_name':'adam','lr':0.0001}),("C3: RMSProp lr0.001 dropout0.25",{'dropout':0.25,'optimizer_name':'rmsprop','lr':0.001}),("C4: Adam dropout0.5",{'dropout':0.5,'optimizer_name':'adam','lr':0.001})]
kf=KFold(n_splits=5,shuffle=True,random_state=SEED)
cv_res=[]
for name,cfg in configs:
    accs=[]
    for tr,va in kf.split(all_feat):
        _,h,_=train_head(all_feat[tr],all_oh[tr],all_feat[va],all_oh[va],dropout=cfg['dropout'],optimizer_name=cfg['optimizer_name'],lr=cfg['lr'],epochs=8,verbose=0)
        accs.append(float(max(h.history['val_accuracy'])))
    mean=float(np.mean(accs)); sd=float(np.std(accs))
    cv_res.append((name,accs,mean,sd)); print(name, f"{mean:.4f} ± {sd:.4f}", accs)
fig,ax=plt.subplots(figsize=(10,5))
means=[m for _,_,m,_ in cv_res]; sds=[s for _,_,_,s in cv_res]; names=[n.split(":")[0] for n,_,_,_ in cv_res]
x=np.arange(len(names)); bars=ax.bar(x, np.array(means)*100, yerr=np.array(sds)*100, capsize=6, color=['steelblue','seagreen','darkorange','crimson'], edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(names,fontsize=8); ax.set_title("Plot 13: 5-Fold Cross-Validation Accuracy (Mean ± SD)"); ax.set_ylabel("mean validation accuracy (%)")
for bar,m,s in zip(bars,means,sds): ax.text(bar.get_x()+bar.get_width()/2, m*100+s*100+0.3, f"{m*100:.1f}±{s*100:.1f}",ha='center',fontsize=8)
ax.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.savefig("exp5_plot_13_cv_accuracy.png",dpi=130); plt.show()
best=max(cv_res,key=lambda x: x[2]); print("Best",best[0],best[2])


## 12. Final Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix
best_cfg=dict(configs[[c[0] for c in configs].index(best[0])][1])
# retrain on full data
_,h,_=train_head(all_feat,all_oh,val_features,y_val_oh,dropout=best_cfg['dropout'],optimizer_name=best_cfg['optimizer_name'],lr=best_cfg['lr'],epochs=12,verbose=0)
final_model=build_head(all_feat.shape[1],NUM_CLASSES,dropout=best_cfg['dropout'])
opt={"adam":tf.keras.optimizers.Adam(best_cfg['lr']),"rmsprop":tf.keras.optimizers.RMSprop(best_cfg['lr'])}[best_cfg['optimizer_name']]
final_model.compile(optimizer=opt,loss='categorical_crossentropy',metrics=['accuracy'])
final_model.fit(all_feat,all_oh,epochs=12,batch_size=32,verbose=0)
probs=final_model.predict(test_features,verbose=0); preds=np.argmax(probs,axis=1)
acc=accuracy_score(test_labels,preds); prec=precision_score(test_labels,preds,average='macro',zero_division=0)
rec=recall_score(test_labels,preds,average='macro',zero_division=0); f1=f1_score(test_labels,preds,average='macro',zero_division=0)
print(f"Test acc {acc:.4f} prec {prec:.4f} rec {rec:.4f} f1 {f1:.4f}")
print(classification_report(test_labels,preds,target_names=BREEDS,digits=4))
cm=confusion_matrix(test_labels,preds)
fig,ax=plt.subplots(figsize=(12,10))
sns.heatmap(cm,cmap='Blues',xticklabels=False,yticklabels=False,ax=ax,cbar=True)
ax.set_title("Plot 14: Confusion Matrix (37 breeds) - Final Model"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("exp5_plot_14_confusion.png",dpi=130); plt.show()
# misclassified
wrong=np.where(preds!=test_labels)[0][:12]
fig,axes=plt.subplots(3,4,figsize=(12,9))
for ax,idx in zip(axes.flat,wrong):
    ax.imshow(test_images[idx]); ax.set_title(f"true: {BREEDS[test_labels[idx]][:12]}\n pred: {BREEDS[preds[idx]][:12]}",fontsize=7); ax.axis('off')
fig.suptitle("Plot 15: Misclassified Oxford-IIIT Pet Images"); plt.tight_layout(); plt.savefig("exp5_plot_15_misclassified.png",dpi=130); plt.show()


## Overall Results

In [ ]:
print("Best CV",best[0],best[2],best[3])
print(f"Final test {acc:.4f} CV mean {best[2]:.4f} SD {best[3]:.4f}")
